In [2]:
# Step 1 - imports and Load the Excel file 

import openpyxl
from datetime import datetime

FILE_PATH = "12515872.xlsx"    
MINUTES_IN_A_DAY = 1440


wb = openpyxl.load_workbook(FILE_PATH, data_only=True)
ws = wb["Daily Log"]  

for row in ws.iter_rows(values_only=True):
    print("\t".join(str(v) for v in row))

My Data, My Code — Daily Log	None	None	None	None	None	None	None	None	None	None	None	None	None
Name:	Sephora Tchinda Fopa	None	None	Registration No.:	12515872	None	None	None	None	None	None	None	None
Course / Section:	CAP 776 / D1P2537	None	None	Month:	August	None	None	None	None	None	None	None	None
All durations in MINUTES. Yellow cells = enter your data. Row 6 is a sample — overwrite it. Rows 2-3: fill your details.	None	None	None	None	None	None	None	None	None	None	None	None	None
Date	Sleep (min)	Fitness (min)	Study (min)	Coding (min)	Class (min)	Classes Attended	Other Activities (min)	Total Tracked (min)	Free/Unaccounted (min)	Day's Feeling	Satisfaction Level	Energy Level	Notes
2026-07-20 00:00:00	420	45	120	90	180	3	30	885	555	Good	Satisfied	Medium	Example row — replace with your own data. Felt a bit tired after evening class.
2026-08-01 00:00:00	360	90	180	120	0	0	300	1050	390	Excellent	Very Satisfied	High	it's weekend so good day
2026-08-02 00:00:00	420	60	240	180	0	0	300	1200	240	E

In [3]:
# Step 2 - Encoding the text responses to numeric values for analysis

feeling_scale = {"EXcellent":5, "Good": 4, "Neutral": 3, "Low": 2, "Stressed": 1}
satisfaction_scale = {"Very Satisfied": 5, "Satisfied": 4, "Neutral": 3, "Unsatisfied": 2, "Very Unsatisfied": 1}
energy_scale = {"High": 5, "Medium": 3, "Low": 1}


In [ ]:
# Step 3 — Read every row as a list skipping bad/blank ones
# Row 6 is the example row in the Excel file. The first 5 rows are headers 
# and instructions, so we skip them since real data starts at row 7. 

START_DATE = datetime(2026, 8, 1).date()
END_DATE = datetime(2026, 9, 21).date()
 
days = []            # each entry will be a dictionary for one valid day
skipped = []         # human-readable notes on every row we couldn't use
invalid_dates = set()  # dates where a row existed but had bad/missing data
 
for row_num in range(7, ws.max_row + 1):
    date_val = ws.cell(row=row_num, column=1).value
    sleep = ws.cell(row=row_num, column=2).value
    fitness = ws.cell(row=row_num, column=3).value
    study = ws.cell(row=row_num, column=4).value
    coding = ws.cell(row=row_num, column=5).value
    class_time = ws.cell(row=row_num, column=6).value
    other = ws.cell(row=row_num, column=8).value
    feeling_text = ws.cell(row=row_num, column=11).value
    satisfaction_text = ws.cell(row=row_num, column=12).value
    energy_text = ws.cell(row=row_num, column=13).value
 
    # if the whole row is empty, just stop looking (end of the data)
    if date_val is None and sleep is None:
        continue
 
    # the date has to be valid and inside the recording period before
    # we can even say WHICH day this row's problems belong to
    if not isinstance(date_val, datetime):
        skipped.append(f"Row {row_num}: missing/invalid date")
        continue
 
    day_date = date_val.date()
    if day_date < START_DATE or day_date > END_DATE:
        skipped.append(f"Row {row_num}: date {day_date} is outside the recording period")
        continue
 
    # now validate the rest of the row's values
    try:
        sleep = float(sleep)
        fitness = float(fitness)
        study = float(study)
        coding = float(coding)
        class_time = float(class_time)
        other = float(other)
 
        feeling = feeling_scale[feeling_text]
        satisfaction = satisfaction_scale[satisfaction_text]
        energy = energy_scale[energy_text]
 
    except (TypeError, ValueError, KeyError) as e:
        invalid_dates.add(day_date)
        skipped.append(f"Row {row_num} ({day_date}): invalid data - {e}")
        continue
 
    total_tracked = sleep + fitness + study + coding + class_time + other
    free_time = MINUTES_IN_A_DAY - total_tracked
 
    days.append({
        "date": day_date,
        "sleep": sleep, "fitness": fitness, "study": study, "coding": coding,
        "class_time": class_time, "other": other,
        "total_tracked": total_tracked, "free_time": free_time,
        "feeling": feeling, "satisfaction": satisfaction, "energy": energy,
    })
 
print(f"Loaded {len(days)} valid days.")
for msg in skipped:
    print(msg)

Loaded 41 valid days.
Row 7 (2026-08-01): invalid data - 'Excellent'
Row 8 (2026-08-02): invalid data - 'Excellent'
Row 10 (2026-08-04): invalid data - 'Excellent'
Row 13 (2026-08-07): invalid data - 'Excellent'
Row 30 (2026-08-24): invalid data - 'Excellent'
Row 33 (2026-08-27): invalid data - 'Excellent'
Row 43 (2026-09-06): invalid data - 'Excellent'
Row 45 (2026-09-08): invalid data - 'Excellent'
Row 46 (2026-09-09): invalid data - 'Excellent'
Row 54 (2026-09-17): invalid data - 'Excellent'
Row 58 (2026-09-21): invalid data - 'Excellent'


In [5]:
# Step 4 - calculate the 8 indices (just sum divided by number of days)

n = len(days)
 
TPI = sum(d["coding"] for d in days) / n
AAI = sum(d["study"] + d["class_time"] for d in days) / n
PhAI = sum(d["fitness"] for d in days) / n
SRI = sum(d["sleep"] for d in days) / n
ABI = sum(d["free_time"] for d in days) / n
TUI = sum(d["total_tracked"] for d in days) / n
EI = sum((d["feeling"] + d["satisfaction"] + d["energy"]) / 3 for d in days) / n
 
expected_days = (END_DATE - START_DATE).days + 1
DCI = (n / expected_days) * 100
 
# ---- Fill in the rest of the "Activity Data Summary" table ----
valid_dates = {d["date"] for d in days}          # dates with a fully usable row
invalid_count = len(invalid_dates)               # rows present but with bad data
missing_count = expected_days - len(valid_dates) - invalid_count  # no usable row at all
 
avg_study = sum(d["study"] for d in days) / n
avg_class = sum(d["class_time"] for d in days) / n
avg_other = sum(d["other"] for d in days) / n
 
print("\n--- Activity Data Summary ---")
print(f"Expected number of days:     {expected_days}")
print(f"Valid days recorded:         {n}")
print(f"Missing days:                {missing_count}")
print(f"Invalid / excluded records:  {invalid_count}")
print(f"Average Study/day:           {avg_study:.2f} min/day")
print(f"Average Class/day:           {avg_class:.2f} min/day")
print(f"Average Other Activities/day: {avg_other:.2f} min/day")
 
print("\n--- Activity Indices ---")
print(f"TPI  (Tech Productivity):  {TPI:.2f} min/day")
print(f"AAI  (Academic Activity):  {AAI:.2f} min/day")
print(f"PhAI (Physical Activity):  {PhAI:.2f} min/day")
print(f"SRI  (Sleep & Recovery):   {SRI:.2f} min/day")
print(f"ABI  (Activity Balance):   {ABI:.2f} min/day")
print(f"TUI  (Time Utilization):   {TUI:.2f} min/day")
print(f"EI   (Experience, 1-5):    {EI:.2f}")
print(f"DCI  (Data Continuity):    {DCI:.2f}%")


--- Activity Data Summary ---
Expected number of days:     52
Valid days recorded:         41
Missing days:                0
Invalid / excluded records:  11
Average Study/day:           136.83 min/day
Average Class/day:           134.15 min/day
Average Other Activities/day: 197.56 min/day

--- Activity Indices ---
TPI  (Tech Productivity):  90.00 min/day
AAI  (Academic Activity):  270.98 min/day
PhAI (Physical Activity):  45.24 min/day
SRI  (Sleep & Recovery):   379.76 min/day
ABI  (Activity Balance):   456.46 min/day
TUI  (Time Utilization):   983.54 min/day
EI   (Experience, 1-5):    3.07
DCI  (Data Continuity):    78.85%


In [6]:
# Step 5 - Personal Activity Index (PAI) is a weighted sum of the other indices.
# The weights can be adjusted based on personal priorities.

TPI_s = TPI / MINUTES_IN_A_DAY * 100
AAI_s = AAI / MINUTES_IN_A_DAY * 100
PhAI_s = PhAI / MINUTES_IN_A_DAY * 100
SRI_s = SRI / MINUTES_IN_A_DAY * 100
TUI_s = TUI / MINUTES_IN_A_DAY * 100
EI_s = EI / 5 * 100
 
PAI = (0.15 * TPI_s + 0.20 * AAI_s + 0.15 * PhAI_s + 0.20 * SRI_s
       + 0.15 * TUI_s + 0.10 * EI_s + 0.05 * DCI)
 
print(f"\nPAI (Personal Activity Index): {PAI:.2f} / 100")


PAI (Personal Activity Index): 30.76 / 100


In [7]:
# Step 6 - correlations between a few pairs of variables
#  A simple manual Pearson correlation — no numpy.

def correlation(list1, list2):
    avg1 = sum(list1) / len(list1)
    avg2 = sum(list2) / len(list2)
    top = sum((a - avg1) * (b - avg2) for a, b in zip(list1, list2))
    bottom1 = sum((a - avg1) ** 2 for a in list1)
    bottom2 = sum((b - avg2) ** 2 for b in list2)
    bottom = (bottom1 * bottom2) ** 0.5
    if bottom == 0:
        return 0
    return top / bottom
 
 
coding_vals = [d["coding"] for d in days]
sleep_vals = [d["sleep"] for d in days]
study_vals = [d["study"] for d in days]
energy_vals = [d["energy"] for d in days]
satisfaction_vals = [d["satisfaction"] for d in days]
 
r_coding_energy = correlation(coding_vals, energy_vals)
r_sleep_energy = correlation(sleep_vals, energy_vals)
r_study_satisfaction = correlation(study_vals, satisfaction_vals)
 
print("\n--- Relationships ---")
print(f"Coding vs Energy:  r = {r_coding_energy:.2f}")
print(f"Sleep vs Energy:   r = {r_sleep_energy:.2f}")
print(f"Study vs Satisfaction: r = {r_study_satisfaction:.2f}")


--- Relationships ---
Coding vs Energy:  r = 0.17
Sleep vs Energy:   r = 0.15
Study vs Satisfaction: r = 0.34
